# Ingest data to Gold zone

In [ ]:
target_table = 'staging.dim_vendor'
set_cols = {'vendor_id', 'vendor_name', 'meomeo'}
pk = 'vendor_id'

updated_cols = set_cols - {pk}
updated_set = [f'{col} = EXCLUDED.{col}' for col in updated_cols]
query = f'''
    insert into {target_table} ({",".join(set_cols)})
    select {",".join(set_cols)}
    from {target_table}_tmp
    on conflict ({pk}) do update
        set {", ".join(updated_set)}
'''
print(query)


    insert into staging.dim_vendor (meomeo,vendor_id,vendor_name)
    select meomeo,vendor_id,vendor_name
    from staging.dim_vendor_tmp
    on conflict (vendor_id) do update
        set meomeo = EXCLUDED.meomeo, vendor_name = EXCLUDED.vendor_name;



In [1]:
import sys

sys.path.append('/home/nblong/Dropbox/Project/ops/ml-ops/src/')

import os
from datetime import datetime, timezone

from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from utils.db_connection import PostgresConnection


In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    # delta lake support
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # key fot S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # read/write delta lake, postgres
    .config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.4.2,'
        'software.amazon.awssdk:bundle:2.29.52,'
        'io.delta:delta-spark_4.1_2.13:4.3.0,'
        'org.postgresql:postgresql:42.7.11,'
    )
    .config("spark.sql.warehouse.dir", "file:///tmp/spark-warehouse")
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/07 13:25:08 WARN Utils: Your hostname, debian-nblong, resolves to a loopback address: 127.0.1.1; using 192.168.36.108 instead (on interface wlp0s20f3)
26/10/07 13:25:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nblong/.ivy2.5.2/cache
The jars for the packages stored in: /home/nblong/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
io.delta#delta-spark_4.1_2.13 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-740c1ea8-b89e-42e6-878c-bf0ec26010d6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.

In [7]:
db_host='localhost'

try:
    # try to read staging table (if existed) and extract watermark (max silver_updated_datetime)
    current_staging_df = (
        spark.read.format("jdbc")
        .option("driver", "org.postgresql.Driver")
        .option("url", f"jdbc:postgresql://{db_host}:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_GOLD_DB_NAME')}")
        .option("dbtable", "public.nyc_taxi_staging")
        .option("user", os.getenv("COM_DB_USER"))
        .option("password", os.getenv("COM_DB_PASSWORD"))
        .load()
    )

    current_staging_df.createOrReplaceTempView('staging')
    watermark = str(spark.sql('select max(silver_updated_datetime) from staging').collect()[0][0])
    if watermark is None:
        _is_existed = False
        watermark = '2000-01-01 00:00:00.0'
    else:
        _is_existed = True
except Exception:
    # if not existed
    _is_existed = False
    print('Please check whether table `public.nyc_taxi_staging` existed before running this job\nReset watermark')
    watermark = '2000-01-01 00:00:00.0'

watermark, _is_existed

('2026-08-21 22:07:13.848787', True)

In [64]:
spark.sql('select max(silver_updated_datetime) from staging').toPandas().head()

,max(silver_updated_datetime)
0,2026-08-14 13:34:45.376317


In [ ]:
batch_start_ts = datetime.now(timezone.utc)
watermark = '2026-08-15 22:07:13.848787'

silver_table = 'silver.nyc_taxi'
gold_staging_df = spark.read.table(silver_table)
gold_staging_df.createOrReplaceTempView('nyc_taxi')
gold_staging_df = spark.sql(f'''
    select * from nyc_taxi
    where silver_created_datetime <= (
        select max(silver_created_datetime)
        from (
            select * from nyc_taxi
            where silver_created_datetime > timestamp '{watermark}'
            order by silver_created_datetime asc
            limit {140_000*3}
        )
    ) and silver_created_datetime > timestamp '{watermark}'
    and silver_created_datetime < '{batch_start_ts!s}'
''')
# gold_staging_df.count()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/07 08:49:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
silver_table = 'silver.nyc_taxi'
gold_staging_df = spark.read.table(silver_table)
gold_staging_df = gold_staging_df.filter((gold_staging_df['year']==2009) & (gold_staging_df['month']==10))

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [ ]:
def upsert_staging_dim(df:DataFrame, target_table:str, db_host:str, set_cols:set[str], pk:str):
    (
        df.select(*set_cols).distinct()
        .write
        .format("jdbc")
        .option("url", f"jdbc:postgresql://{db_host}:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_GOLD_DB_NAME')}")
        .option("dbtable", f'{target_table}_tmp')
        .option("user", os.getenv('COM_DB_USER'))
        .option("password", os.getenv('COM_DB_PASSWORD'))
        .option("driver", "org.postgresql.Driver")
        .mode("overwrite")
        .save()
    )

    updated_cols = set_cols - {pk}
    updated_set = [f'{col}: EXCLUDED.{col}' for col in updated_cols]

    query = f'''
        insert into {target_table} ({",".join(set_cols)})
        select {",".join(set_cols)}
        from {target_table}_tmp
        on conflict {pk} do update
            set {",".join(updated_set)}
    '''

    pg = PostgresConnection(host=db_host, dbname=os.getenv('COM_GOLD_DB_NAME'))
    with pg as conn, conn.cursor() as cur:
        cur.execute(query)
        conn.commit()

upsert_staging_dim(gold_staging_df, 'dim_vendor', 'localhost', {'vendor'}, 'vendor')

- `ingest_gold_staging.py` kéo data từ Silver zone vào Postgres (bảng `nyc_taxi_staging`)
- Trước khi kéo, sẽ truncate data. Đây sẽ là task đầu tiên trong DAG gold. Phải pass task này mới đi đến các task khác

In [22]:
gold_staging_df.filter((gold_staging_df['year']==2009) & (gold_staging_df['month']==10)).count()

139386

In [21]:
gold_staging_df.select('year', 'month').drop_duplicates().show()

+----+-----+
|year|month|
+----+-----+
|2009|   10|
|2009|    9|
|2009|   11|
|2009|   12|
+----+-----+



In [23]:
host='localhost'

pg = PostgresConnection(host, dbname='gold')
with pg as conn:
    with conn.cursor() as cur:
        cur.execute('create schema if not exists staging')
    conn.commit()

(
    gold_staging_df.filter((gold_staging_df['year']==2009) & (gold_staging_df['month']==10))
    .write
    .format("jdbc")
    .option("url", f"jdbc:postgresql://{host}:5432/{os.getenv('COM_GOLD_DB_NAME')}")
    .option("dbtable", "staging.nyc_taxi")
    .option("user", os.getenv('COM_DB_USER'))
    .option("password", os.getenv('COM_DB_PASSWORD'))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)

## Chia các bảng `dim, fact`

- Chia cột vào các bảng
- Viết logic để upsert vào các bảng (mặc định là các bảng đã được tạo trước đó rồi)
- Chi tiết schema: `./sql/gold_schema.sql`

In [ ]:
dim_vendor = df_silver.select()